# Notebook 06 – Error Analysis

**Purpose:** Understand WHERE and WHY the model fails.

**Structure:**
1. Segment test set by prediction outcome (TP / TN / FP / FN)
2. Profile False Negatives vs True Positives
3. Segment-level performance table and bar chart
4. SHAP explanations for the 5 most confidently wrong predictions
5. Prediction confidence distribution
6. Fairness check across gender and senior citizen groups
7. Learning curve
8. Limitations

In [1]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix
from sklearn.model_selection import learning_curve, StratifiedKFold

from data_utils import load_raw, make_splits
from features  import ChurnFeatureEngineer
from evaluate  import find_best_threshold
from explain   import get_shap_values, plot_shap_waterfall

FIG_DIR    = Path('../reports/figures')
RES_DIR    = Path('../reports/results')
MODELS_DIR = Path('../models')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['savefig.dpi'] = 300

def savefig(name):
    plt.savefig(FIG_DIR / name, dpi=300, bbox_inches='tight')
    plt.show()

## 1. Load Data, Pipeline & Predictions

In [2]:
df = load_raw()
X_train, X_val, X_test, y_train, y_val, y_test = make_splits(df)

artifact  = joblib.load(MODELS_DIR / 'best_pipeline.joblib')
pipeline  = artifact['pipeline']
best_name = artifact['model_name']

y_prob = pipeline.predict_proba(X_test)[:, 1]
best_t, _ = find_best_threshold(y_test.values, y_prob, metric='f2')
y_pred    = (y_prob >= best_t).astype(int)

print(f'Model: {best_name}  |  Threshold: {best_t}  |  Test size: {len(X_test)}')

Model: LightGBM_Tuned  |  Threshold: 0.32  |  Test size: 1409


## 2. Segment by Outcome

In [3]:
fe = ChurnFeatureEngineer()
X_test_fe = fe.fit_transform(X_test.copy())

analysis = X_test_fe.copy()
analysis['true_label'] = y_test.values
analysis['pred_label'] = y_pred
analysis['pred_prob']  = y_prob

def label_outcome(row):
    t, p = row['true_label'], row['pred_label']
    if t == 1 and p == 1: return 'TP'
    if t == 0 and p == 0: return 'TN'
    if t == 1 and p == 0: return 'FN'
    return 'FP'

analysis['outcome'] = analysis.apply(label_outcome, axis=1)
counts = analysis['outcome'].value_counts()
print(counts)

fn_df = analysis[analysis['outcome'] == 'FN']
tp_df = analysis[analysis['outcome'] == 'TP']
fp_df = analysis[analysis['outcome'] == 'FP']
tn_df = analysis[analysis['outcome'] == 'TN']
print(f'\nFN rate: {len(fn_df)/(len(fn_df)+len(tp_df)):.3f}  FP rate: {len(fp_df)/(len(fp_df)+len(tn_df)):.3f}')

outcome
TN    619
FP    416
TP    340
FN     34
Name: count, dtype: int64

FN rate: 0.091  FP rate: 0.402


## 3. False Negative Profile vs True Positive

In [4]:
compare_cols = ['tenure','MonthlyCharges','TotalCharges','num_services','avg_monthly_spend','charge_gap']
compare = pd.DataFrame({
    'FN (missed)': fn_df[compare_cols].mean(),
    'TP (caught)': tp_df[compare_cols].mean(),
    'FP (false alarm)': fp_df[compare_cols].mean(),
    'TN (correct)': tn_df[compare_cols].mean(),
}).round(2)
print(compare.to_string())
compare.to_csv(RES_DIR / 'error_profile.csv')

colors = {'FN':'#D65F5F','TP':'#4878CF','FP':'#F0A500','TN':'#6ACC65'}
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(compare_cols):
    for outcome, subset in [('FN', fn_df), ('TP', tp_df)]:
        subset[col].plot.kde(ax=axes[i], label=outcome, color=colors[outcome], linewidth=2)
    axes[i].set_title(col, fontweight='bold')
    axes[i].legend(fontsize=8)
plt.suptitle('Feature Distributions: FN vs TP', fontsize=13, fontweight='bold')
plt.tight_layout()
savefig('26_fn_vs_tp_distributions.png')

                   FN (missed)  TP (caught)  FP (false alarm)  TN (correct)
tenure                   41.65        16.11             24.28         46.45
MonthlyCharges           66.44        76.19             71.26         53.55
TotalCharges           3327.62      1416.31           2041.40       2821.78
num_services              2.82         1.68              1.84          2.21
avg_monthly_spend        63.85        63.48             64.90         51.93
charge_gap                2.59        12.71              6.36          1.63


**Insight:** FNs have longer tenure and lower MonthlyCharges than TPs.
The model is less confident about longer-tenured customers on month-to-month
contracts — a segment that warrants additional features (e.g., service call frequency).

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (subset, title) in zip(axes, [
    (fn_df, 'False Negatives'), (tp_df, 'True Positives')
]):
    subset['Contract'].value_counts(normalize=True).plot(
        kind='bar', ax=ax, color='#D65F5F', alpha=0.85, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Proportion')
    ax.tick_params(axis='x', rotation=20)
plt.suptitle('Contract Type in Error Groups', fontweight='bold')
plt.tight_layout()
savefig('27_error_contract_type.png')

## 4. Prediction Confidence by Outcome

In [6]:
fig, ax = plt.subplots(figsize=(9, 4))
for outcome, subset, color in [
    ('FN', fn_df, '#D65F5F'), ('FP', fp_df, '#F0A500'),
    ('TP', tp_df, '#4878CF'), ('TN', tn_df, '#6ACC65')
]:
    if len(subset) > 1:
        subset['pred_prob'].plot.kde(ax=ax, label=outcome, color=color, linewidth=2)
ax.axvline(best_t, color='black', linestyle='--', linewidth=1.5, label=f'Threshold={best_t}')
ax.set_xlabel('Predicted Probability of Churn')
ax.set_title('Prediction Confidence by Outcome', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
savefig('28_confidence_by_outcome.png')

## 5. Segment-Level Performance

In [7]:
segment_cols = ['Contract', 'tenure_group', 'InternetService', 'SeniorCitizen', 'gender']
rows = []

for seg_col in segment_cols:
    for seg_val in analysis[seg_col].unique():
        mask = analysis[seg_col] == seg_val
        sub  = analysis[mask]
        if len(sub) < 10:
            continue
        yt = sub['true_label'].values
        yp = sub['pred_label'].values
        rows.append({
            'segment_col':  seg_col,
            'segment_val':  str(seg_val),
            'n':            len(sub),
            'churn_rate':   round(yt.mean(), 3),
            'precision':    round(precision_score(yt, yp, zero_division=0), 3),
            'recall':       round(recall_score(yt, yp, zero_division=0), 3),
            'f1':           round(f1_score(yt, yp, zero_division=0), 3),
        })

seg_df = pd.DataFrame(rows).sort_values(['segment_col', 'recall'])
seg_df.to_csv(RES_DIR / 'segment_performance.csv', index=False)
print(seg_df.to_string(index=False))

    segment_col    segment_val    n  churn_rate  precision  recall    f1
       Contract       Two year  339       0.041      0.200   0.071 0.105
       Contract       One year  284       0.127      0.343   0.667 0.453
       Contract Month-to-month  786       0.412      0.463   0.972 0.627
InternetService             No  303       0.073      0.261   0.545 0.353
InternetService            DSL  482       0.178      0.361   0.849 0.507
InternetService    Fiber optic  624       0.426      0.502   0.959 0.659
  SeniorCitizen              0 1170       0.232      0.425   0.890 0.575
  SeniorCitizen              1  239       0.427      0.527   0.961 0.681
         gender         Female  714       0.290      0.482   0.903 0.629
         gender           Male  695       0.240      0.416   0.916 0.572
   tenure_group            60+  292       0.058      0.196   0.529 0.286
   tenure_group          49-60  164       0.183      0.362   0.700 0.477
   tenure_group          25-48  318       0.223    

In [8]:
# Flag underperforming segments (recall < overall recall)
overall_recall = recall_score(y_test.values, y_pred, zero_division=0)
underperform   = seg_df[seg_df['recall'] < overall_recall - 0.05]
print(f'Overall recall: {overall_recall:.3f}')
print(f'\nUnderperforming segments (recall < {overall_recall - 0.05:.3f}):')
print(underperform[['segment_col','segment_val','n','recall','f1']].to_string(index=False))

Overall recall: 0.909

Underperforming segments (recall < 0.859):
    segment_col segment_val   n  recall    f1
       Contract    Two year 339   0.071 0.105
       Contract    One year 284   0.667 0.453
InternetService          No 303   0.545 0.353
InternetService         DSL 482   0.849 0.507
   tenure_group         60+ 292   0.529 0.286
   tenure_group       49-60 164   0.700 0.477


In [9]:
# Bar chart: recall by Contract and tenure_group
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, seg_col in zip(axes, ['Contract', 'tenure_group']):
    sub = seg_df[seg_df['segment_col'] == seg_col].set_index('segment_val')
    sub[['precision','recall','f1']].plot(kind='bar', ax=ax, alpha=0.85, edgecolor='white')
    ax.axhline(overall_recall, color='black', linestyle='--', linewidth=1,
               label=f'Overall recall={overall_recall:.2f}')
    ax.set_title(f'Performance by {seg_col}', fontweight='bold')
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=20)
    ax.legend(fontsize=7)

plt.suptitle('Segment-Level Model Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
savefig('29_segment_performance.png')

**Insight:** The model underperforms on the **60+ month tenure group** and
**two-year contract** segment — both low-churn segments where the few churners
are harder to distinguish. These segments may benefit from separate sub-models
or additional features.

## 6. SHAP for the 5 Most Confidently Wrong Predictions

In [10]:
# Most confidently wrong = highest predicted prob among FNs
# + lowest predicted prob among FPs
fn_sorted = fn_df.sort_values('pred_prob', ascending=False)
fp_sorted = fp_df.sort_values('pred_prob', ascending=True)

# Top 3 confident FNs + top 2 confident FPs
wrong_indices = list(fn_sorted.index[:3]) + list(fp_sorted.index[:2])
wrong_labels  = (
    [f'FN_rank{i+1}_prob{fn_sorted.iloc[i]["pred_prob"]:.2f}' for i in range(min(3, len(fn_sorted)))] +
    [f'FP_rank{i+1}_prob{fp_sorted.iloc[i]["pred_prob"]:.2f}' for i in range(min(2, len(fp_sorted)))]
)

print('Most confidently wrong predictions:')
for idx, label in zip(wrong_indices, wrong_labels):
    row = analysis.loc[idx]
    print(f'  {label}  true={int(row["true_label"])}  pred={int(row["pred_label"])}')

Most confidently wrong predictions:
  FN_rank1_prob0.29  true=1  pred=0
  FN_rank2_prob0.29  true=1  pred=0
  FN_rank3_prob0.28  true=1  pred=0
  FP_rank1_prob0.32  true=0  pred=1
  FP_rank2_prob0.32  true=0  pred=1


In [11]:
for idx, label in zip(wrong_indices, wrong_labels):
    # Map back to original X_test index
    X_single = X_test.loc[[idx]]
    try:
        path = plot_shap_waterfall(pipeline, X_single, best_name, label=label)
        print(f'Saved: {path.name}')
    except Exception as e:
        print(f'Skipped {label}: {e}')

Saved: 24_shap_waterfall_LightGBM_Tuned_FN_rank1_prob0.29.png


Saved: 24_shap_waterfall_LightGBM_Tuned_FN_rank2_prob0.29.png


Saved: 24_shap_waterfall_LightGBM_Tuned_FN_rank3_prob0.28.png


Saved: 24_shap_waterfall_LightGBM_Tuned_FP_rank1_prob0.32.png


Saved: 24_shap_waterfall_LightGBM_Tuned_FP_rank2_prob0.32.png


## 7. Fairness Check: Recall & FPR by Gender and Senior Citizen

In [12]:
fairness_rows = []
for col in ['gender', 'SeniorCitizen']:
    for val in analysis[col].unique():
        mask = analysis[col] == val
        sub  = analysis[mask]
        yt   = sub['true_label'].values
        yp   = sub['pred_label'].values
        cm   = confusion_matrix(yt, yp, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr    = fp / (fp + tn) if (fp + tn) > 0 else 0
        fairness_rows.append({
            'group_col': col, 'group_val': str(val),
            'n': len(sub), 'churn_rate': round(yt.mean(), 3),
            'recall': round(recall, 3), 'FPR': round(fpr, 3),
            'f1': round(f1_score(yt, yp, zero_division=0), 3),
        })

fairness_df = pd.DataFrame(fairness_rows)
fairness_df.to_csv(RES_DIR / 'fairness_check.csv', index=False)
print(fairness_df.to_string(index=False))

    group_col group_val    n  churn_rate  recall   FPR    f1
       gender    Female  714       0.290   0.903 0.396 0.629
       gender      Male  695       0.240   0.916 0.407 0.572
SeniorCitizen         0 1170       0.232   0.890 0.365 0.575
SeniorCitizen         1  239       0.427   0.961 0.642 0.681


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ['gender', 'SeniorCitizen']):
    sub = fairness_df[fairness_df['group_col'] == col].set_index('group_val')
    sub[['recall', 'FPR']].plot(kind='bar', ax=ax, alpha=0.85, edgecolor='white',
                                 color=['#4878CF', '#D65F5F'])
    ax.set_title(f'Fairness: {col}', fontweight='bold')
    ax.set_ylabel('Rate')
    ax.tick_params(axis='x', rotation=0)
    ax.set_ylim(0, 1)

plt.suptitle('Fairness Check: Recall & FPR by Demographic Group', fontsize=12, fontweight='bold')
plt.tight_layout()
savefig('30_fairness_check.png')

**Fairness interpretation:**
- If recall differs substantially between Male/Female or Senior/Non-Senior,
  the model may be systematically missing churners in one demographic group.
- A higher FPR for one group means that group receives more unnecessary
  retention offers — a form of disparate impact.
- Note: gender and senior citizen status have low Cramér's V with churn
  (EDA notebook), so large disparities are unlikely but must be verified.

## 8. Learning Curve

In [14]:
from sklearn.model_selection import learning_curve, StratifiedKFold

# Combine train + val for the learning curve (more data points)
import pandas as pd
X_tv = pd.concat([X_train, X_val], axis=0)
y_tv = pd.concat([y_train, y_val], axis=0)

train_sizes, train_scores, val_scores = learning_curve(
    pipeline,
    X_tv, y_tv,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='average_precision',
    n_jobs=-1,
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_mean, 'o-', color='#4878CF', label='Train PR-AUC')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color='#4878CF')
ax.plot(train_sizes, val_mean, 'o-', color='#D65F5F', label='CV Val PR-AUC')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.15, color='#D65F5F')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('PR-AUC')
ax.set_title(f'Learning Curve – {best_name}', fontweight='bold')
ax.legend()
plt.tight_layout()
savefig('31_learning_curve.png')

**Interpretation:**
- If the validation curve is still rising at the maximum training size,
  more data would likely improve performance.
- If the train and val curves have converged, the model is in the
  bias-dominated regime — more data will not help; better features or
  a more complex model are needed.
- A large gap between train and val curves indicates overfitting.

## 9. Save Error Tables

In [15]:
fn_df.to_csv(RES_DIR / 'false_negatives.csv', index=False)
fp_df.to_csv(RES_DIR / 'false_positives.csv', index=False)
print('Saved: false_negatives.csv, false_positives.csv')
print(f'FN count: {len(fn_df)}  FP count: {len(fp_df)}')

Saved: false_negatives.csv, false_positives.csv
FN count: 34  FP count: 416


## 10. Limitations

The following limitations should be acknowledged in the report:

| Limitation | Detail | Impact |
|------------|--------|--------|
| **Dataset size** | 7,043 rows is small for ML. The model may not generalise to larger, more diverse customer bases. | Moderate |
| **Single company** | Data from one unnamed telecom provider. Churn drivers may differ across companies, regions, and markets. | High |
| **No temporal information** | A static snapshot with no timestamps. Churn is inherently a time-series problem; survival analysis would be more appropriate. | High |
| **No behavioural data** | Missing features like call frequency, data usage, support ticket history, and website activity — all known churn predictors. | High |
| **Possible label noise** | 'Churn' is binary; customers who downgraded or paused are not distinguished from those who fully cancelled. | Low–Moderate |
| **No causal claims** | SHAP values show correlation, not causation. A month-to-month contract does not *cause* churn; both may be driven by a latent variable (e.g., price sensitivity). | Conceptual |
| **Class imbalance** | 26.5% churn is moderate imbalance. Metrics and thresholds are sensitive to the assumed cost ratio (FN=$500, FP=$50). | Moderate |
| **Potential bias** | If the training data reflects historical biases (e.g., certain demographics were offered fewer retention deals), the model may perpetuate those biases. | Moderate |

In [16]:
print('Notebook 06 complete.')
print(f'Figures saved: 26–31 in reports/figures/')
print(f'CSVs saved: error_profile, segment_performance, fairness_check, false_negatives, false_positives')

Notebook 06 complete.
Figures saved: 26–31 in reports/figures/
CSVs saved: error_profile, segment_performance, fairness_check, false_negatives, false_positives
